In [ ]:
import pandas as pd
import pymysql
from pymongo import MongoClient
import psycopg2
import matplotlib.pyplot as plt
import time as tm
from folium import plugins
import json
import urllib.request
from math import radians, cos, sin, asin, sqrt
import matplotlib
import numpy as np
import datetime
from datetime import datetime
from tqdm import tqdm
from dask import delayed


In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


db = client.get_database('telematics_db')

db2 = client2.get_database('telematics_db')





def gettime(x):
    try:
        return pd.Timestamp(x).timestamp()
    except:
        return None

In [ ]:
def haversine(L1,L2):
    """
    Calculate the great circle distance between two points
    on the earth (specified in decimal degrees)
    """

    lon1=L1[0]
    lat1=L1[1]
    lon2=L2[0]
    lat2=L2[1]
    #print(lon1,lat1,lon2,lat2)
    # convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])

    # haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    r = 6371000 # Radius of earth in kilometers. Use 3956 for miles
    return c * r

In [ ]:
csvwriter = open('distance_in_no_info_june1.csv','a')



def no_info(device_history, count, complete_data):
    print('.')
    device_history=pd.DataFrame.from_dict(device_history,orient='index').transpose()
        
    try:
        
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder={}&timeCondition={}".format(device_history.iloc[0]['vehicleId'],int(device_history.iloc[0]['fromTime']),int(device_history.iloc[0]['toTime']),'asc','')
        print(url)
        data=json.load(urllib.request.urlopen(url))
        
        data1=pd.DataFrame(data['data'][str(device_history.iloc[0]['vehicleId'])])        
        
        
        print(' API SE DATA AA GAYA, WITH LENGTH ' + str(len(data1)) )
          
        
        
    except:
        
        print('api error')
    
    if len(data1)>1:
        
        
        
        try:
            
            data1['loc2']=data1.location.shift(1)
            data1=data1[1:]
            loc_list=data1[['location','loc2']].values.tolist()

            dis=[]
            for loc in loc_list:
                y=haversine(loc[0],loc[1])
                dis.append(y)
            data1['distance']=dis
            data1['no_info_id']=[data1.iloc[0]['_id']]*len(data1)
            #print("data1 tak chal gaya")
            data2=pd.DataFrame([[device_history.iloc[0]['vehicleId'],
                                     device_history.iloc[0]['_id'],
                                     data1.distance.sum(),
                                     data1[data1['distance']>500]['distance'].sum(),
                                     len(data1),
                                     len(data1[data1['distance']>500]),
                                     (device_history.iloc[0]['toTime']-device_history.iloc[0]['fromTime']),
                                     len(data1[data1['speed']<6]),
                                     len(data1[data1['speed']<6]),
                                     (device_history.iloc[0]['toTime']-device_history.iloc[0]['fromTime'])/30,
                                   device_history.iloc[0]['suggestion'],
                                datetime.fromtimestamp(device_history.iloc[0]['fromTime']+5.5*3600).date()]],
                                   
                                   columns=['vehicle_id',
                                            'no_info_id',
                                            'total_distance',
                                            'total_jump',
                                            'total_pings',
                                            'jump_pings',
                                            'total_time',
                                            'stop_pings',
                                            'running_pings',
                                            'min_expected_pings',
                                           'suggestion',
                                           'date_of_no_info'])
            #print("data2 bhi ban gaya")
            #complete_data=pd.concat([complete_data,data1])
            #analysed_data=pd.concat([analysed_data,data2])

            row = data2.to_csv(header=False)
            print(row)
            csvwriter.write(row)
            csvwriter.flush()


        except Exception as e:
            print(e)
    else:
        try:
            data2=pd.DataFrame([[device_history.iloc[0]['vehicleId'],
                                 device_history.iloc[0]['_id'],
                                 0,
                                 0,
                                 1,
                                 1,
                                 (device_history.iloc[0]['toTime']-device_history.iloc[0]['fromTime']),
                                 1,
                                 1,
                                 (device_history.iloc[0]['toTime']-device_history.iloc[0]['fromTime'])/30,
                               device_history.iloc[0]['suggestion'],
                                datetime.fromtimestamp(device_history.iloc[0]['fromTime']+5.5*3600).date()]],
                               columns=['vehicle_id',
                                        'no_info_id',
                                        'total_distance',
                                        'total_jump',
                                        'total_pings',
                                        'jump_pings',
                                        'total_time',
                                        'stop_pings',
                                        'running_pings',
                                        'min_expected_pings',
                                       'suggestion',
                                       'date_of_no_info'])
            
            row = data2.to_csv(header=False)
            print(row)
            csvwriter.write(row)
            csvwriter.flush()

        except Exception as e:
            print(e)
            
        
    count=count+1
    print(count)
            
    return data2, count, complete_data




In [ ]:
delayed(no_info)

In [ ]:
device_history=pd.DataFrame(list(db.deviceHistory.find({"fromTime":{"$lte":1559347200,'$gte':(1559347200-86400)}})))
#device_history=device_history[device_history['suggestion']!='Battery disconnected']
device_history.toTime.fillna(tm.time(),inplace=True)
#converting the DataFrame to list of dictionaries

try:
    back_log=pd.read_csv("distance_in_no_info_june1.csv")

    set1=set(back_log['_id'])
except:
    set1=set()
set2=set(device_history['_id'])

device_history=device_history[device_history['_id'].isin(set2-set1)]
device_history=device_history.to_dict('records')



%%time
complete_data=pd.DataFrame()
analysed_data=pd.DataFrame()
count=len(pd.read_csv("diatance_in_no_info.csv"))
for i in device_history:
    _, analysed_data, complete_data =no_info(i, analysed_data, complete_data)

In [ ]:

from multiprocessing.pool import ThreadPool as Pool
pool = Pool() 
complete_data=pd.DataFrame()
analysed_data=pd.DataFrame()
count=len(pd.read_csv("distance_in_no_info_june1.csv"))

for i in device_history:
    pool.apply_async(no_info, (i,count,complete_data))
#pool.map(no_info, device_history)
#pd.concat(lengths)
pool.close()
pool.join()

In [ ]:
a=pd.read_csv("distance_in_no_info_june1.csv")

In [ ]:
backup=a

In [ ]:
a

In [ ]:
a[a['total distance']<10000]

In [ ]:
a[a['total distance']]=a['total distance']/1000

In [ ]:
a[a['total distance']<5][a['sugestion']!='Battery disconnected']

In [ ]:
a[a['sugestion']=='Installer visit is required']

In [ ]:
matplotlib.pyplot.hist(a['distance'])

In [ ]:
a

In [ ]:
a['ping_diffrence']=a['min_expected pings']-a['total_pings']

In [ ]:
a['bad_device']=a

In [ ]:
vehicle_details=pd.read_sql("select * from vehicle_details",con=postgres)

In [ ]:
len(vehicle_details[vehicle_details['sim_phone_number'].isna()])

In [ ]:
a[a['total distance']>5][a['sugestion']=='Battery disconnected']

In [ ]:
a.columns

In [ ]:
a=pd.merge(a,vehicle_details,on='vehicle_id',how='left')

In [ ]:
a['sim_phone_number'].isna().value_counts()

In [ ]:
a['sim_phone_number'].fillna(0,inplace=True)

In [ ]:
a['digs']=a['sim_phone_number'].apply(str)

In [ ]:
a['sim_phone_number']=a['sim_phone_number'].apply(str)

In [ ]:
len(a.iloc[8796]['sim_phone_number'])

In [ ]:
a['digs']=a['sim_phone_number'].apply(lambda x: len(x))

In [ ]:
a[a['total distance']>5][a['digs']==10]

In [ ]:
len(a[a['digs']==15]['vehicle_id'].unique())

In [ ]:
len(a[a['digs']==10]['vehicle_id'].unique())

In [ ]:
#a.iloc[0]['live_time']
vehicle_details[vehicle_details['live_time']<datetime(2019,6,1)][vehicle_details['vehicle_state']=='LIVE'][vehicle_details['sim_phone_number'].isna()]

In [ ]:
vehicle_details['sim_phone_number'].fillna(0,inplace=True)

In [ ]:
vehicle_details['sim_phone_number']=vehicle_details['sim_phone_number'].apply(str)

In [ ]:
vehicle_details['digs']=vehicle_details['sim_phone_number'].apply(len)

In [ ]:
vehicle_details[vehicle_details['live_time']<datetime(2019,6,1)][vehicle_details['vehicle_state']=='LIVE'][vehicle_details['digs']==10]      
                                                                                                          
                                                                                                          

In [ ]:
our_devices=['GT06N','WETRACK2','WETRACK-LITE','C100','TL1000','TL250','WETRACK800','AW100','WP30L','GRLAIS','AT50','WP30L','FMB920','FMB207','FMB150','FMB001','LC100_SOS','WETRACK2_SOS','AT4','ATLANTA_OBD','BHARAT101','TRANSYNC_AIS_140','GAGAN']                           


In [ ]:
our_device_list=[]
for i, row in device_models.iterrows():
    if row['name'] in our_devices:
        our_device_list.append('yes')
    else:
        our_device_list.append('no')
device_models['our_device']=our_device_list

In [ ]:
device_models=pd.read_sql("select * from device_models",con=mysql_cn)

In [ ]:
device_models

In [ ]:
a=pd.merge(a,device_models,left_on='model_name',right_on='name',how='left')

In [ ]:
a[a['our_device']=='yes'][a['sub_type_installation']=='SIM_CHANGE']

In [ ]:
a['time_based_no_info']=a.apply(lambda row: 
                                
                                if (row['total distance']<2)&(row['total_pings']==row['stop_pings']):
                                
                                
                                 return 'True'
                                else)

In [ ]:
del a['id_x']

In [ ]:
a.columns

In [ ]:
time_based=pd.DataFrame(list(db.deviceHistory.find({'fromTime':{'$gte':1559415600,'$lte':(1559415600+36000)},'toTime':{'$gte':(1559415600+36000),'$lte':(1559415600+36000+7*3600)}})))

In [ ]:
time_based
